# ENCO6067 — Jegadeesh & Titman (1993) Momentum Replication: Data Preparation

This notebook consolidates the code from `code/00_profile_data.py` and `code/01_build_panel.py`
so you can inspect the results cell by cell. Make sure the raw data is in `../raw data/`
(or set the `RAW_DATA_DIR` environment variable).

- Cells 1-4: data profiling (monthly stock file + Fama-French factors)
- Cells 5-6: build the clean monthly panel and validate it


In [ ]:
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

# Locate the repo root (which holds config.py) regardless of the kernel's cwd.
for p in [Path.cwd(), Path.cwd().parent]:
    if (p / "config.py").exists():
        sys.path.insert(0, str(p))
        break

import config

config.ensure_dirs()
print("RAW_DATA_DIR  =", config.RAW_DATA_DIR)
print("MONTHLY_STOCK =", config.MONTHLY_STOCK_CSV)
print("FF_FACTORS    =", config.FF_FACTORS_CSV)
print("DAILY_STOCK   =", config.DAILY_STOCK_CSV)
print("sample period =", config.START_YYYYMM, "..", config.END_YYYYMM)


## Step 0 — Monthly stock data profile

Chunked read (to avoid loading the 1.2 GB file at once): row count, date range, number of
securities, missingness of key fields, and the return distribution.

In [ ]:
cols = ["PERMNO", "MthCalDt", "MthRet", "MthRetx", "MthCap",
         "PrimaryExch", "SecurityType", "ShareType", "USIncFlg", "MthDelFlg"]
dtypes = {"MthRet": "float32", "MthRetx": "float32", "MthCap": "float64"}

n_rows = 0
n_ret_nn = n_retx_nn = n_cap_nn = 0
permnos = set()
in_sample_permnos = set()
in_sample_rows = 0
d_min = d_max = None
rets = []
exch = Counter(); stype = Counter(); shtype = Counter()

reader = pd.read_csv(config.MONTHLY_STOCK_CSV, usecols=cols, dtype=dtypes,
                     parse_dates=["MthCalDt"], chunksize=500_000)
for chunk in reader:
    n_rows += len(chunk)
    permnos.update(chunk["PERMNO"].dropna().astype(int).tolist())
    n_ret_nn += int(chunk["MthRet"].notna().sum())
    n_retx_nn += int(chunk["MthRetx"].notna().sum())
    n_cap_nn += int(chunk["MthCap"].notna().sum())
    rets.append(chunk["MthRet"].dropna().to_numpy(dtype="float32"))
    dt = chunk["MthCalDt"]
    lo, hi = dt.min(), dt.max()
    d_min = lo if d_min is None or lo < d_min else d_min
    d_max = hi if d_max is None or hi > d_max else d_max
    for c, counter in [("PrimaryExch", exch), ("SecurityType", stype), ("ShareType", shtype)]:
        counter.update(chunk[c].astype(str).value_counts(dropna=False).to_dict())
    in_ = dt.dt.year.between(config.START_YEAR, config.END_YEAR)
    in_sample_rows += int(in_.sum())
    in_sample_permnos.update(chunk.loc[in_, "PERMNO"].dropna().astype(int).tolist())

print(f"rows         : {n_rows:,}")
print(f"date range   : {d_min:%Y-%m} .. {d_max:%Y-%m}")
print(f"unique PERMNO: {len(permnos):,}")
for c, nn in [("MthRet", n_ret_nn), ("MthRetx", n_retx_nn), ("MthCap", n_cap_nn)]:
    print(f"{c:10s} non-null: {nn:>12,}  ({nn/n_rows:.1%})")

r = np.concatenate(rets)
print(f"MthRet mean/median: {r.mean():.4f} / {np.median(r):.4f}")
print(f"MthRet min / max  : {r.min():.4f} / {r.max():.4f}")
print(f"MthRet |r|<=1 frac: {(np.abs(r) <= 1).mean():.2%}   |r|>2 rows: {(np.abs(r) > 2).sum():,}")

for name, counter in [("PrimaryExch", exch), ("SecurityType", stype), ("ShareType", shtype)]:
    print(f"\n{name} top values: {dict(counter.most_common(8))}")

print(f"\nrows in {config.START_YEAR}-{config.END_YEAR}: {in_sample_rows:,} "
      f"({in_sample_rows / n_rows:.1%}); unique PERMNO = {len(in_sample_permnos):,}")


## Step 0b — Fama-French factors and risk-free rate

`F-F factors and RF.csv` starts with 3 prose lines and a header on line 5, followed by
`YYYYMM` monthly rows (in percent) and an annual section at the end. Keep only the 6-digit
monthly rows.

In [ ]:
ff = pd.read_csv(config.FF_FACTORS_CSV, skiprows=4)
ff = ff.rename(columns={ff.columns[0]: "YYYYMM"})
ff["YYYYMM"] = ff["YYYYMM"].astype(str)
monthly = ff[ff["YYYYMM"].str.fullmatch("[0-9]{6}")].copy()
monthly["YYYYMM"] = monthly["YYYYMM"].astype(int)
print(f"monthly rows: {len(monthly):,}  ({monthly['YYYYMM'].min()} .. {monthly['YYYYMM'].max()})")
print(f"columns     : {list(monthly.columns)}")

sub = monthly[monthly["YYYYMM"].between(config.START_YYYYMM, config.END_YYYYMM)]
rf = sub["RF"].astype(float) / 100.0
print(f"RF {config.START_YEAR}-{config.END_YEAR} (decimal, monthly): mean {rf.mean():.6f}, median {rf.median():.6f}")


## Step 1 — Build the clean monthly panel

Sample filters (identical to `code/01_build_panel.py`):

1. `PrimaryExch` in {N, A, Q} — NYSE / AMEX / NASDAQ
2. `SecurityType == "EQTY"` — common equity only (drops funds/ETFs)
3. `ShareType == "NS"` — common shares
4. `MthRet` not null
5. Sample period 2000-01 to 2025-12 (the file already covers exactly this range)

In [ ]:
COLS = ["PERMNO", "MthCalDt", "MthRet", "MthCap", "PrimaryExch", "SecurityType", "ShareType"]
DTYPES = {"MthRet": "float32", "MthCap": "float64"}
EXCHANGES = {"N", "A", "Q"}

n_in = n_exch = n_eqty = n_share = n_ret = 0
chunks = []
reader = pd.read_csv(config.MONTHLY_STOCK_CSV, usecols=COLS, dtype=DTYPES,
                     parse_dates=["MthCalDt"], chunksize=1_000_000)
for chunk in reader:
    n_in += len(chunk)
    chunk = chunk[chunk["PrimaryExch"].isin(EXCHANGES)]; n_exch += len(chunk)
    chunk = chunk[chunk["SecurityType"] == "EQTY"];       n_eqty += len(chunk)
    chunk = chunk[chunk["ShareType"] == "NS"];            n_share += len(chunk)
    chunk = chunk[chunk["MthRet"].notna()];               n_ret += len(chunk)
    chunks.append(chunk[["PERMNO", "MthCalDt", "MthRet", "MthCap"]])

panel = pd.concat(chunks, ignore_index=True).sort_values(["PERMNO", "MthCalDt"]).reset_index(drop=True)
panel["PERMNO"] = panel["PERMNO"].astype("int32")

print(f"read {n_in:,}  ->  exchange N/A/Q {n_exch:,}  ->  EQTY {n_eqty:,}  ->  NS {n_share:,}  ->  ret nonna {n_ret:,}")
print(f"panel: {len(panel):,} rows, {panel['PERMNO'].nunique():,} PERMNOs, "
      f"{panel['MthCalDt'].dt.to_period('M').nunique():,} months")
print(f"MthRet mean/median: {panel['MthRet'].mean():.4f} / {panel['MthRet'].median():.4f}")

out = config.DATA_DIR / "monthly_panel.parquet"
panel.to_parquet(out, index=False)
print("saved ->", out)
panel.head()


## Step 1b — Panel validation

Check the monthly cross-section sizes and the market-cap distribution.

In [ ]:
p = panel.copy()
p["ym"] = p["MthCalDt"].dt.to_period("M")
cnt = p.groupby("ym")["PERMNO"].nunique()
print("stocks per month: min", cnt.min(), " median", int(cnt.median()), " max", cnt.max())
print()
print(p[["PERMNO", "MthCalDt", "MthRet", "MthCap"]].head(3).to_string(index=False))
print()
print("MthCap summary (thousands of USD):")
print(p["MthCap"].describe().to_string())
